In [1]:
!pip install xplique

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/ConceptualizingConceptDrift/')

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

import torch
import numpy as np

from sklearn.decomposition import NMF, non_negative_factorization
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from itertools import combinations
from scipy.optimize import linear_sum_assignment
from sklearn.metrics.pairwise import cosine_similarity

import random

from experiment_helpers.helper_function import *
from experiment_helpers.driftLocalizer import Localizer
from text_helpers.CraftText import CraftText, CraftTextCombined, full_text_activations
from text_helpers.CustomBertModel import CustomBertForSequenceClassification

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

device = 'cuda'

model_checkpoint = "rttl-ai/bert-base-uncased-yelp-reviews"
model = CustomBertForSequenceClassification.from_pretrained(model_checkpoint, num_labels=5)
model = model.eval().to(device)
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

yelp = load_dataset("Yelp/yelp_review_full")

# yelp-texts contain \n instead of actual newlines
texts = [t.replace("\\n", " ") for t in yelp["train"]["text"]]
labels = np.array(yelp["train"]["label"])

keys = list(np.unique(labels))


def cosine_hungarian_loss(V1, V2):
    assert V1.shape == V2.shape
    assert len(V1.shape) == 2
    cost_matrix = 1 - cosine_similarity(V1, V2)
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    loss = cost_matrix[row_ind, col_ind].sum()
    return float(loss / len(V1))


def stability(v_drift_list):
    scores = []
    n = len(v_drift_list)
    for i in range(n):
        for j in range(i+1, n):
            scores.append(cosine_hungarian_loss(v_drift_list[i], v_drift_list[j]))
    return np.mean(scores), np.std(scores)


def stability_across(result_dict, value_range):
    scores = []
    for val_a, val_b in combinations(value_range, 2):
        for va in result_dict[val_a]['v_drift_list']:
            for vb in result_dict[val_b]['v_drift_list']:
                scores.append(cosine_hungarian_loss(va, vb))
    return np.mean(scores), np.std(scores)

[nltk_data] Downloading package punkt to /root/nltk_data...


[nltk_data]   Unzipping tokenizers/punkt.zip.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

...

yelp_review_full/train-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

yelp_review_full/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 23.5MB            

yelp_review_full/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
import os
import random


np.random.seed(42)
random.seed(42)

patch_mode_range = ["sentence", "window"]
runs = 25
n_concepts = 10
win_size = 15
stride = 10

result_dict_patchmode = {patch_mode: {
    "label_maps": [],
    "drift_ratios": [],
    "drift_localizer": [],
    "one_local_l": [],
    "one_local_l_lp" : [],
    "recon_single" : [],
    "recon_single_lp" : [],
    "recon_all" : [],
    "recon_all_lp" : [],
    "v_drift_list" : [],
    } for patch_mode in patch_mode_range}


for patch_mode in patch_mode_range:
    for run in range(runs):
        sample_ids = np.random.choice(len(texts), 500, False)
        sample_texts = [texts[i] for i in sample_ids]

        keys_shuffled = keys.copy()
        random.shuffle(keys_shuffled)

        initial_labels = [0, 1, 2]
        random.shuffle(initial_labels)

        label_map = {keys_shuffled[i]: initial_labels[i] for i in range(3)}

        for i in range(3, len(keys_shuffled)):
            label_map[keys_shuffled[i]] = random.randint(0, 2)

        result_dict_patchmode[patch_mode]["label_maps"].append(label_map)

        labels_mapped = np.array([label_map[class_id] for class_id in labels])

        drift_labels = labels_mapped[sample_ids]

        label_2_idx = np.where(drift_labels == 2)[0]
        y_mixed = drift_labels.copy()
        y_mixed[label_2_idx] = np.random.choice([0, 1], size=len(label_2_idx))

        sample_labels = y_mixed

        result_dict_patchmode[patch_mode]["drift_ratios"].append({"BD": len(np.where(drift_labels == 0)[0]),
                                                        "AD": len(np.where(drift_labels == 1)[0]),
                                                        "Both": len(np.where(drift_labels == 2)[0])})

        patch_act = full_text_activations(sample_texts, model, tokenizer, device=device)
        train_labels = sample_labels

        bd_indices = np.where(sample_labels != 1)[0]
        ad_indices = np.where(sample_labels != 0)[0]

        bd_texts = [sample_texts[i] for i in bd_indices]
        bd_labels = [sample_labels[i] for i in bd_indices]
        ad_texts = [sample_texts[i] for i in ad_indices]
        ad_labels = [sample_labels[i] for i in ad_indices]

        bd_fit = CraftText(model_wrapper=model, tokenizer=tokenizer, num_concepts=n_concepts,
                            patch_mode=patch_mode, win_size=win_size, stride=stride, device=device)
        print("Fitting Unsupervised Craft....")
        bd_crops, bd_crops_u, bd_w = bd_fit.fit(bd_texts, bd_labels)

        ad_fit = CraftText(model_wrapper=model, tokenizer=tokenizer, num_concepts=n_concepts,
                            patch_mode=patch_mode, win_size=win_size, stride=stride, device=device)
        print("Fitting Unsupervised Craft....")
        ad_crops, ad_crops_u, ad_w = ad_fit.fit(ad_texts, ad_labels)

        drift_basis = np.vstack([bd_w, ad_w])

        result_dict_patchmode[patch_mode]["v_drift_list"].append(drift_basis.copy())

        drift_craft = CraftTextCombined(model_wrapper=model, tokenizer=tokenizer, basis=drift_basis,
                                         patch_mode=patch_mode, win_size=win_size, stride=stride, device=device)
        print("Fitting Craft....")
        drift_craft.transform_all(sample_texts, sample_labels)

        X_clean = patch_act
        y_clean = train_labels

        localizer_model = Localizer()

        X_train_clean, X_test_clean, y_train, y_test = \
            train_test_split(X_clean, y_clean, train_size=0.7, random_state=42)

        print('Fitting Random Forest classifier...')
        localizer_model.fit(X_train_clean, y_train)
        print('Fitting complete.')

        localizer_bin_preds = localizer_model.l_predict(X_test_clean)
        result_dict_patchmode[patch_mode]["drift_localizer"].append(accuracy_score(localizer_bin_preds, y_test))

        drift_imp = np.round(estimate_importance_l(localizer_model, drift_craft, drift_basis, X_train_clean), 3)

        image_drift_imp_l = [estimate_importance_helper_l(drift_craft, localizer_model, drift_basis,
                                                      image, class_of_interest=localizer_bin_preds[i])
                                   for i, image in enumerate(X_test_clean)]

        localizer_bin_train_preds = localizer_model.l_predict(X_train_clean)
        image_drift_imp_l_train = [estimate_importance_helper_l(drift_craft, localizer_model, drift_basis,
                                                      image, class_of_interest=localizer_bin_train_preds[i])
                                   for i, image in enumerate(X_train_clean)]
        concept_dist = concept_counter(image_drift_imp_l_train, localizer_bin_train_preds)

        result_dict_patchmode[patch_mode]['one_local_l'].append(
            local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=1, labels=y_test))
        result_dict_patchmode[patch_mode]['one_local_l_lp'].append(
            local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=1, labels=localizer_bin_preds))

        recon_single = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=1)
        result_dict_patchmode[patch_mode]['recon_single'].append(
            accuracy_score(localizer_model.l_predict(recon_single), y_test))
        result_dict_patchmode[patch_mode]['recon_single_lp'].append(
            accuracy_score(localizer_model.l_predict(recon_single), localizer_bin_preds))

        recon_all = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=2*n_concepts)
        result_dict_patchmode[patch_mode]['recon_all'].append(
            accuracy_score(localizer_model.l_predict(recon_all), y_test))
        result_dict_patchmode[patch_mode]['recon_all_lp'].append(
            accuracy_score(localizer_model.l_predict(recon_all), localizer_bin_preds))

        print(f"patch_mode = {patch_mode} , run {run+1}")


Backup geladen: {'sentence': 25, 'window': 25} Runs vorhanden


In [ ]:
import csv

method_names = [
    "drift_localizer",
    "one_local_l",
    "one_local_l_lp",
    "recon_single",
    "recon_single_lp",
    "recon_all",
    "recon_all_lp",
]

with open('/content/drive/MyDrive/results/results_patchmode_yelp.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['patch_mode', 'Method'] + [f'Run_{i+1}' for i in range(runs)])
    for patch_mode in patch_mode_range:
        for method_name in method_names:
            row = [patch_mode, method_name] + result_dict_patchmode[patch_mode][method_name]
            writer.writerow(row)

with open('/content/drive/MyDrive/results/stability_patchmode_yelp.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['patch_mode', 'stability_mean', 'stability_std'])
    for patch_mode in patch_mode_range:
        loss_mean, loss_std = stability(result_dict_patchmode[patch_mode]['v_drift_list'])
        writer.writerow([patch_mode, loss_mean, loss_std])
    cross_mean, cross_std = stability_across(result_dict_patchmode, patch_mode_range)
    writer.writerow(['cross_all_values', cross_mean, cross_std])

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/results/results_patchmode_yelp.csv')

stats = {}
for patch_mode in df['patch_mode'].unique():
    for method in df['Method'].unique():
        subset = df[(df['patch_mode'] == patch_mode) & (df['Method'] == method)]
        accuracies = subset.drop(['patch_mode', 'Method'], axis=1).values.flatten().astype(float)
        mean = np.mean(accuracies)
        std = np.std(accuracies)
        stats[(patch_mode, method)] = (mean, std)

for key, (mean, std) in stats.items():
    print(f"{key[0]:10s} {key[1]:25s} mean={mean:.3f}  std={std:.3f}")

sentence   drift_localizer           mean=0.779  std=0.076
sentence   one_local_l               mean=0.765  std=0.074
sentence   one_local_l_lp            mean=0.901  std=0.060
sentence   recon_single              mean=0.763  std=0.071
sentence   recon_single_lp           mean=0.921  std=0.048
sentence   recon_all                 mean=0.783  std=0.071
sentence   recon_all_lp              mean=0.963  std=0.021
window     drift_localizer           mean=0.760  std=0.063
window     one_local_l               mean=0.749  std=0.067
window     one_local_l_lp            mean=0.886  std=0.065
window     recon_single              mean=0.750  std=0.076
window     recon_single_lp           mean=0.905  std=0.047
window     recon_all                 mean=0.762  std=0.064
window     recon_all_lp              mean=0.963  std=0.024


In [ ]:
import os

import random

np.random.seed(42)
random.seed(42)

win_size_range = [5, 10, 15, 20, 30]
runs = 25
n_concepts = 10
patch_mode = "window"

result_dict_winsize = {win_size: {
    "label_maps": [],
    "drift_ratios": [],
    "drift_localizer": [],
    "one_local_l": [],
    "one_local_l_lp" : [],
    "recon_single" : [],
    "recon_single_lp" : [],
    "recon_all" : [],
    "recon_all_lp" : [],
    "v_drift_list" : [],
    } for win_size in win_size_range}

for win_size in win_size_range:
    stride = max(1, int(win_size * 0.67))
    for run in range(runs):
        sample_ids = np.random.choice(len(texts), 500, False)
        sample_texts = [texts[i] for i in sample_ids]

        keys_shuffled = keys.copy()
        random.shuffle(keys_shuffled)

        initial_labels = [0, 1, 2]
        random.shuffle(initial_labels)

        label_map = {keys_shuffled[i]: initial_labels[i] for i in range(3)}

        for i in range(3, len(keys_shuffled)):
            label_map[keys_shuffled[i]] = random.randint(0, 2)

        result_dict_winsize[win_size]["label_maps"].append(label_map)

        labels_mapped = np.array([label_map[class_id] for class_id in labels])

        drift_labels = labels_mapped[sample_ids]

        label_2_idx = np.where(drift_labels == 2)[0]
        y_mixed = drift_labels.copy()
        y_mixed[label_2_idx] = np.random.choice([0, 1], size=len(label_2_idx))

        sample_labels = y_mixed

        result_dict_winsize[win_size]["drift_ratios"].append({"BD": len(np.where(drift_labels == 0)[0]),
                                                        "AD": len(np.where(drift_labels == 1)[0]),
                                                        "Both": len(np.where(drift_labels == 2)[0])})

        patch_act = full_text_activations(sample_texts, model, tokenizer, device=device)
        train_labels = sample_labels

        bd_indices = np.where(sample_labels != 1)[0]
        ad_indices = np.where(sample_labels != 0)[0]

        bd_texts = [sample_texts[i] for i in bd_indices]
        bd_labels = [sample_labels[i] for i in bd_indices]
        ad_texts = [sample_texts[i] for i in ad_indices]
        ad_labels = [sample_labels[i] for i in ad_indices]

        bd_fit = CraftText(model_wrapper=model, tokenizer=tokenizer, num_concepts=n_concepts,
                            patch_mode=patch_mode, win_size=win_size, stride=stride, device=device)
        print("Fitting Unsupervised Craft....")
        bd_crops, bd_crops_u, bd_w = bd_fit.fit(bd_texts, bd_labels)

        ad_fit = CraftText(model_wrapper=model, tokenizer=tokenizer, num_concepts=n_concepts,
                            patch_mode=patch_mode, win_size=win_size, stride=stride, device=device)
        print("Fitting Unsupervised Craft....")
        ad_crops, ad_crops_u, ad_w = ad_fit.fit(ad_texts, ad_labels)

        drift_basis = np.vstack([bd_w, ad_w])

        result_dict_winsize[win_size]["v_drift_list"].append(drift_basis.copy())

        drift_craft = CraftTextCombined(model_wrapper=model, tokenizer=tokenizer, basis=drift_basis,
                                         patch_mode=patch_mode, win_size=win_size, stride=stride, device=device)
        print("Fitting Craft....")
        drift_craft.transform_all(sample_texts, sample_labels)

        X_clean = patch_act
        y_clean = train_labels

        localizer_model = Localizer()

        X_train_clean, X_test_clean, y_train, y_test = \
            train_test_split(X_clean, y_clean, train_size=0.7, random_state=42)

        print('Fitting Random Forest classifier...')
        localizer_model.fit(X_train_clean, y_train)
        print('Fitting complete.')

        localizer_bin_preds = localizer_model.l_predict(X_test_clean)
        result_dict_winsize[win_size]["drift_localizer"].append(accuracy_score(localizer_bin_preds, y_test))

        drift_imp = np.round(estimate_importance_l(localizer_model, drift_craft, drift_basis, X_train_clean), 3)

        image_drift_imp_l = [estimate_importance_helper_l(drift_craft, localizer_model, drift_basis,
                                                      image, class_of_interest=localizer_bin_preds[i])
                                   for i, image in enumerate(X_test_clean)]

        localizer_bin_train_preds = localizer_model.l_predict(X_train_clean)
        image_drift_imp_l_train = [estimate_importance_helper_l(drift_craft, localizer_model, drift_basis,
                                                      image, class_of_interest=localizer_bin_train_preds[i])
                                   for i, image in enumerate(X_train_clean)]
        concept_dist = concept_counter(image_drift_imp_l_train, localizer_bin_train_preds)

        result_dict_winsize[win_size]['one_local_l'].append(
            local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=1, labels=y_test))
        result_dict_winsize[win_size]['one_local_l_lp'].append(
            local_imp_concepts_probability(concept_dist, image_drift_imp_l, num=1, labels=localizer_bin_preds))

        recon_single = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=1)
        result_dict_winsize[win_size]['recon_single'].append(
            accuracy_score(localizer_model.l_predict(recon_single), y_test))
        result_dict_winsize[win_size]['recon_single_lp'].append(
            accuracy_score(localizer_model.l_predict(recon_single), localizer_bin_preds))

        recon_all = reconstruct_inputs(X_test_clean, image_drift_imp_l, drift_basis, num_concepts=2*n_concepts)
        result_dict_winsize[win_size]['recon_all'].append(
            accuracy_score(localizer_model.l_predict(recon_all), y_test))
        result_dict_winsize[win_size]['recon_all_lp'].append(
            accuracy_score(localizer_model.l_predict(recon_all), localizer_bin_preds))

        print(f"win_size = {win_size} (stride={stride}) , run {run+1}")


Backup geladen: {5: 25, 10: 25, 15: 25, 20: 25, 30: 7} Runs vorhanden


Fitting Unsupervised Craft....


Fitting Unsupervised Craft....


Fitting Craft....


Fitting Random Forest classifier...
Determine optimal parameters using cross validation


...

/usr/local/lib/python3.13/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


/usr/local/lib/python3.13/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


/usr/local/lib/python3.13/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklea

/usr/local/lib/python3.13/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklea

win_size = 30 (stride=20) , run 25


In [ ]:
import csv

method_names = [
    "drift_localizer",
    "one_local_l",
    "one_local_l_lp",
    "recon_single",
    "recon_single_lp",
    "recon_all",
    "recon_all_lp",
]

with open('/content/drive/MyDrive/results/results_winsize_yelp.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['win_size', 'Method'] + [f'Run_{i+1}' for i in range(runs)])
    for win_size in win_size_range:
        for method_name in method_names:
            row = [win_size, method_name] + result_dict_winsize[win_size][method_name]
            writer.writerow(row)

with open('/content/drive/MyDrive/results/stability_winsize_yelp.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['win_size', 'stability_mean', 'stability_std'])
    for win_size in win_size_range:
        loss_mean, loss_std = stability(result_dict_winsize[win_size]['v_drift_list'])
        writer.writerow([win_size, loss_mean, loss_std])
    cross_mean, cross_std = stability_across(result_dict_winsize, win_size_range)
    writer.writerow(['cross_all_values', cross_mean, cross_std])

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/results/results_winsize_yelp.csv')

stats = {}
for win_size in df['win_size'].unique():
    for method in df['Method'].unique():
        subset = df[(df['win_size'] == win_size) & (df['Method'] == method)]
        accuracies = subset.drop(['win_size', 'Method'], axis=1).values.flatten().astype(float)
        mean = np.mean(accuracies)
        std = np.std(accuracies)
        stats[(win_size, method)] = (mean, std)

for key, (mean, std) in stats.items():
    print(f"{key[0]:5} {key[1]:25s} mean={mean:.3f}  std={std:.3f}")

    5 drift_localizer           mean=0.778  std=0.076
    5 one_local_l               mean=0.769  std=0.075
    5 one_local_l_lp            mean=0.900  std=0.059
    5 recon_single              mean=0.769  std=0.076
    5 recon_single_lp           mean=0.913  std=0.052
    5 recon_all                 mean=0.783  std=0.072
    5 recon_all_lp              mean=0.956  std=0.028
   10 drift_localizer           mean=0.760  std=0.064
   10 one_local_l               mean=0.756  std=0.072
   10 one_local_l_lp            mean=0.899  std=0.055
   10 recon_single              mean=0.757  std=0.071
   10 recon_single_lp           mean=0.899  std=0.053
   10 recon_all                 mean=0.766  std=0.063
   10 recon_all_lp              mean=0.961  std=0.023
   15 drift_localizer           mean=0.767  std=0.070
   15 one_local_l               mean=0.750  std=0.070
   15 one_local_l_lp            mean=0.897  std=0.057
   15 recon_single              mean=0.741  std=0.084
   15 recon_single_lp       